# EchoNet-Dynamic Baseline: R(2+1)D-18 Video Regression

Runs the modular `R(2+1)D-18` 1-channel grey-scale `(1, 32, 112, 112)` baseline on Kaggle.
- Attach your **EchoNet-Dynamic** dataset under **+ Add Input**.
- Attach your code dataset (containing `config.py`, `dataset.py`, `metrics.py`, `trainer.py`, `train.py`, `models/`) under **+ Add Input**, or upload them to `/kaggle/working/`.
- Enable **GPU T4 x2** or **GPU P100** in **Session options**.

In [ ]:
import sys
from pathlib import Path
import torch

print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Locate project files in /kaggle/input or /kaggle/working
project_dir = None
for candidate in [Path('/kaggle/working'), *Path('/kaggle/input').rglob('train.py')]:
    p = candidate if candidate.is_dir() else candidate.parent
    if (p / 'train.py').exists() and (p / 'models' / 'r2plus1d.py').exists():
        project_dir = p
        break

if project_dir is None:
    raise FileNotFoundError("Could not find train.py + models/r2plus1d.py in /kaggle/working or /kaggle/input.")

print(f"Using project directory: {project_dir}")
%cd {project_dir}

In [ ]:
# Run Baseline (R(2+1)D-18) training
!python train.py --model r2plus1d_18 --epochs 45 --batch_size 16 --lr 1e-4

# Run Mamba (GapSSMMamba) training (uncomment to train Mamba as well)
# !python train.py --model mamba --epochs 45 --batch_size 16 --lr 1e-4

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

hist_path = Path('/kaggle/working/outputs/training_history.csv')
if hist_path.exists():
    df = pd.read_csv(hist_path)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(df['epoch'], df['train_mse'], label='Train MSE')
    ax[0].plot(df['epoch'], df['val_mse'], label='Val MSE')
    ax[0].set_title('MSE Loss'); ax[0].set_xlabel('Epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)

    ax[1].plot(df['epoch'], df['train_mae'], label='Train MAE')
    ax[1].plot(df['epoch'], df['val_mae'], label='Val MAE')
    ax[1].set_title('MAE (EF %)'); ax[1].set_xlabel('Epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/kaggle/working/outputs/training_curves.png', dpi=150)
    plt.show()
else:
    print('training_history.csv not found yet.')